# Ingestão de Dados de Solicitações (Bronze)

Este notebook reúne dados de solicitações, coletados mensalmente nos diretórios do volume bronze, e estrutura tudo em uma única tabela Delta. Os dados são lidos de arquivos `dados.jsonl` presentes em cada pasta mensal. A abordagem permite futuras expansões, reprocessamentos ou auditorias por período.


## Fluxo dos Dados

* Origem: `/Volumes/cat_mobilidade_urbana_bronze/db_frota/volumes/solicitações/<ano-mes>/dados.jsonl`
* Destino: Tabela Delta `cat_mobilidade_urbana_bronze.db_frota.solicitacoes_bronze`


---

## Etapas:
1. Descoberta das pastas mensais e arquivos
2. Leitura dos dados de todos os meses
3. União das informações em um DataFrame único
4. Persistência em uma tabela Delta


O processo pode ser facilmente adaptado para novas fontes e datas.


## 1. Descoberta das pastas e arquivos

Buscamos todas as subpastas mensais e verificamos a existência do arquivo `dados.jsonl` em cada uma delas. Isso garante robustez para reprocessamento.


In [0]:
import os

BASE_PATH = "/Volumes/cat_mobilidade_urbana_bronze/db_frota/volumes/solicitações"
# Lista todos os diretórios mês
meses = [d for d in os.listdir(BASE_PATH) if os.path.isdir(os.path.join(BASE_PATH, d))]

# Lista todos os arquivos dados.jsonl em cada mês
arquivos = []
for mes in meses:
    caminho_arquivo = os.path.join(BASE_PATH, mes, "dados.jsonl")
    if os.path.exists(caminho_arquivo):
        arquivos.append(caminho_arquivo)

print("Arquivos encontrados:", arquivos)


## 2. Leitura dos dados de todos os meses

Utilizamos Spark para ler cada arquivo `dados.jsonl` encontrado, montando uma lista de DataFrames.


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
dfs = [spark.read.format("json").option("multiLine", False).load(f) for f in arquivos]
print(f"DataFrames lidos: {len(dfs)}")


## 3. União dos dados em um DataFrame único

Todas as informações mensais são unidas para consolidação e análise.


In [0]:
if dfs:
    df_all = dfs[0]
    for df in dfs[1:]:
        df_all = df_all.unionByName(df)
else:
    print("Nenhum arquivo dados.jsonl encontrado nos volumes de solicitações.")


## 4. Salvando o resultado na tabela Delta

Persistimos o DataFrame final consolidado na tabela bronze, permitindo análises e consultas futuras e facilitando processos downstream.


In [0]:
if dfs:
    df_all.write.format("delta").mode("overwrite").saveAsTable("cat_mobilidade_urbana_bronze.db_frota.tb_solicitacoes")
    print("Tabela bronze criada ou atualizada com sucesso.")
else:
    print("Nada foi persistido, pois nenhum arquivo foi encontrado.")
